# Oil Price Shocks and Texas Employment  

## Regression Analysis

**Group:** Anthony Bauer, Jackson Daniel, Logan Averill  

This notebook performs the regression analysis for the project.


## Research Question

Do changes in WTI oil prices predict county-level employment growth in Texas energy-related industries, and does this relationship differ after major oil price shocks? (2010–2025)


## Purpose of This Notebook

The goal of this section is to estimate regression models that quantify the relationship between oil prices and employment growth. Building on the ETL and EDA steps, we move from descriptive analysis to a more formal statistical framework.


## Workflow

1. Load cleaned dataset from ETL pipeline
2. Define expected relationships (prior)
3. Estimate baseline regression models
4. Add controls to improve the model
5. Implement Difference-in-Differences (DiD) analysis
6. Interpret results in plain English
7. Discuss limitations and implications  

All steps are written to be fully reproducible.

# 0) Setup

In [3]:
# Setup — Load Data (Reproducible)

import os
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Load dataset from project structure
base_path = os.path.abspath("..")
df = pd.read_csv(os.path.join(base_path, "data_clean", "final_dataset.csv"))

# Quick check
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (9295, 7)


,area_fips,year,qtr,emplvl,emplvl_growth,WTISPLC,TXUR
0,48001,2010,2,1668.0,-0.017668,77.890000,8.200000
1,48001,2010,3,1699.0,0.018585,76.166667,8.133333
2,48001,2010,4,1773.0,0.043555,85.026667,8.266667
3,48001,2011,1,1758.0,-0.008460,93.980000,8.166667
4,48001,2011,2,1892.0,0.076223,102.553333,8.200000


# 1) OLS Regression/DiD and Interpretations

## Expected Relationships (Prior)

Before running any regression models, we outline what we expect to see based on basic economic reasoning.

We expect **oil prices (WTISPLC)** to have a **positive relationship** with employment growth. When oil prices are higher, energy companies are more profitable and are more likely to increase production, which should lead to more hiring and higher employment growth.

We expect the **unemployment rate (TXUR)** to have a **negative relationship** with employment growth. Higher unemployment usually reflects weaker economic conditions, which would be associated with lower or negative employment growth.

If the regression results do not match these expectations, it likely suggests an issue with the data, model setup, or missing variables rather than a completely new economic finding.

In [4]:
# Baseline Regression: Oil Price → Employment Growth

# Define variables
X = df["WTISPLC"]
y = df["emplvl_growth"]

# Add constant
X = sm.add_constant(X)

# Run regression
model1 = sm.OLS(y, X).fit()

# Extract key results 
coef = model1.params["WTISPLC"]
pval = model1.pvalues["WTISPLC"]
r2 = model1.rsquared

print("Coefficient (Oil Price):", round(coef, 4))
print("P-value:", round(pval, 4))
print("R-squared:", round(r2, 4))

Coefficient (Oil Price): 0.0016
P-value: 0.0
R-squared: 0.014


## Baseline Regression Interpretation

The coefficient on oil prices is **0.0016**, which means that a one-dollar increase in oil prices is associated with about a **0.16 percentage point increase** in employment growth. This suggests a positive relationship between oil prices and employment growth.

The p-value is essentially zero, indicating that this relationship is **statistically significant**, meaning it is unlikely to be due to random chance.

However, the R-squared is only **0.014**, which means that oil prices explain just **1.4% of the variation** in employment growth. This is very low and suggests that oil prices alone do not do a good job of explaining changes in employment.

Overall, the result is **consistent with our expectations** in terms of direction, since higher oil prices are associated with higher employment growth. However, the relationship is quite weak, indicating that other factors play a much larger role in determining employment changes.

## Regression with Controls

To improve the model, we include the unemployment rate as a control variable. This allows us to isolate the relationship between oil prices and employment growth while accounting for overall economic conditions.

By adding unemployment, we can better understand whether oil prices still have an effect on employment growth after controlling for broader labor market trends.

### Prior (continued)

We continue to expect oil prices to have a positive relationship with employment growth. After controlling for unemployment, we expect this relationship to remain positive. We also expect unemployment to have a negative relationship with employment growth, reflecting broader economic conditions.

In [5]:
# Regression with Controls: Oil Price + Unemployment

# Define variables
X = df[["WTISPLC", "TXUR"]]
y = df["emplvl_growth"]

# Add constant
X = sm.add_constant(X)

# Run regression
model2 = sm.OLS(y, X).fit()

# Extract results
coef_oil = model2.params["WTISPLC"]
pval_oil = model2.pvalues["WTISPLC"]

coef_unemp = model2.params["TXUR"]
pval_unemp = model2.pvalues["TXUR"]

r2 = model2.rsquared

print("Oil Price Coefficient:", round(coef_oil, 4))
print("Oil Price P-value:", round(pval_oil, 4))

print("Unemployment Coefficient:", round(coef_unemp, 4))
print("Unemployment P-value:", round(pval_unemp, 4))

print("R-squared:", round(r2, 4))

Oil Price Coefficient: 0.0016
Oil Price P-value: 0.0
Unemployment Coefficient: -0.0014
Unemployment P-value: 0.4113
R-squared: 0.0141


## Regression with Controls Interpretation

After adding unemployment as a control variable, the coefficient on oil prices is **0.0016**. This means that, holding unemployment constant, a one-dollar increase in oil prices is associated with about a **0.16 percentage point increase** in employment growth. This is the same as the baseline model, suggesting that the relationship between oil prices and employment growth is stable.

The unemployment coefficient is **-0.0014**, which indicates that higher unemployment is associated with slightly lower employment growth. This matches our expectations in terms of direction. However, the p-value is **0.4113**, meaning this relationship is **not statistically significant**, so we cannot confidently say that unemployment has a meaningful effect in this model.

The R-squared is **0.0141**, which is only slightly higher than before. This means that adding unemployment does not significantly improve the model’s ability to explain variation in employment growth.

Overall, the results are mostly consistent with our expectations. Oil prices continue to show a positive and significant relationship with employment growth, while unemployment has the expected negative sign but does not appear to be an important predictor in this model. This suggests that other factors not included here are likely driving most of the changes in employment growth.

## Difference-in-Differences (DiD) Approach

To better understand the impact of oil price shocks on employment growth, we use a Difference-in-Differences (DiD) approach. This method compares outcomes before and after a major oil price shock.

We define a shock period based on a large decline in oil prices and examine how employment growth changes after this event. This allows us to move closer to identifying the effect of oil price shocks, rather than just simple correlations.

While this approach does not fully establish causality, it provides a more structured way to analyze how employment responds to major changes in oil prices over time.

## Expected Relationship (DiD Prior)

Before running the DiD model, we consider how we expect oil price shocks to affect the relationship between oil prices and employment growth.

We expect that after a major oil price shock (such as the 2020 drop), the relationship between oil prices and employment growth may become **stronger**. This is because sharp declines in oil prices can lead to layoffs and reduced activity in energy-related industries, making employment more sensitive to changes in oil prices during recovery periods.

Alternatively, it is also possible that the relationship weakens after the shock if firms adjust their behavior or if other economic factors dominate.

Overall, we expect the interaction term to be **positive**, indicating that the effect of oil prices on employment growth increases after the shock.

In [7]:
# Create post indicator for the 2020 shock period
df["post"] = (df["year"] >= 2020).astype(int)

# Check that it worked
df[["year", "post"]].drop_duplicates().sort_values("year")

,year,post
0,2010,0
3,2011,0
7,2012,0
11,2013,0
15,2014,0
19,2015,0
23,2016,0
27,2017,0
31,2018,0
35,2019,0


In [8]:
# DiD Regression

# Interaction term
df["oil_post"] = df["WTISPLC"] * df["post"]

# Define variables
X = df[["WTISPLC", "post", "oil_post"]]
y = df["emplvl_growth"]

# Add constant
X = sm.add_constant(X)

# Run regression
model_did = sm.OLS(y, X).fit()

# Extract results
coef_oil = model_did.params["WTISPLC"]
coef_post = model_did.params["post"]
coef_interaction = model_did.params["oil_post"]

pval_interaction = model_did.pvalues["oil_post"]

r2 = model_did.rsquared

print("Oil Coefficient:", round(coef_oil, 4))
print("Post Coefficient:", round(coef_post, 4))
print("Interaction (DiD) Coefficient:", round(coef_interaction, 4))
print("Interaction P-value:", round(pval_interaction, 4))
print("R-squared:", round(r2, 4))

Oil Coefficient: 0.0016
Post Coefficient: -0.0315
Interaction (DiD) Coefficient: 0.0001
Interaction P-value: 0.722
R-squared: 0.0156


## Difference-in-Differences Interpretation

The interaction term between oil prices and the post-2020 period is **0.0001**, which suggests that the relationship between oil prices and employment growth becomes slightly more positive after the oil price shock. However, the effect is extremely small.

The p-value for the interaction term is **0.722**, which indicates that this change is **not statistically significant**. This means we cannot confidently say that the relationship between oil prices and employment growth is different after 2020 compared to before.

The coefficient on oil prices remains positive (**0.0016**), which is consistent with earlier models and suggests that higher oil prices are associated with higher employment growth. The post variable has a coefficient of **-0.0315**, indicating that, on average, employment growth was lower after 2020, which likely reflects the broader economic impact of the COVID-19 period.

The R-squared is **0.0156**, which is still quite low, meaning the model explains only a small portion of the variation in employment growth.

Overall, the results do not provide strong evidence that the oil price shock in 2020 changed the relationship between oil prices and employment growth. While oil prices continue to have a positive association with employment growth, the effect does not appear to be significantly different after the shock.

# Final Conclusion

This project examined whether changes in WTI oil prices are associated with employment growth in Texas energy-related industries, and whether this relationship changes after major oil price shocks. Using data from 2010 to 2025, we analyzed this relationship through exploratory data analysis, regression models, and a Difference-in-Differences (DiD) approach.

The results consistently show that oil prices have a **positive relationship** with employment growth. Across all regression models, higher oil prices are associated with higher employment growth, which aligns with basic economic ideas about the energy sector. However, the effect is relatively small, and the models have low explanatory power, suggesting that oil prices alone do not explain most of the variation in employment.

When adding unemployment as a control variable, the results did not change much. Oil prices remained a significant predictor, while unemployment was not statistically significant. This indicates that broader economic conditions, at least as measured by unemployment, do not substantially improve the model in this case.

The DiD analysis tested whether the relationship between oil prices and employment growth changed after the 2020 oil price shock. The results show that the interaction effect is not statistically significant, meaning there is no strong evidence that the relationship changed after the shock. While employment growth was generally lower after 2020, this likely reflects broader economic disruptions rather than a change in how oil prices affect employment.

There are several limitations to this analysis. The models have low R-squared values, indicating that many important factors affecting employment are not included. The data is aggregated at the county level, which may hide local variation. Additionally, oil prices and unemployment are measured at broader levels and may not fully capture local economic conditions. Finally, the analysis is not causal, and the DiD approach used here is simplified without a clear treatment and control group.

Overall, the findings suggest that oil prices do have a meaningful but limited relationship with employment growth in Texas energy-related industries. While higher oil prices are generally associated with better employment outcomes, they are only one part of a much larger set of factors influencing employment.